# Named Entity Recognition with Synthetic Data

**Notebook 1: Getting started with our base model**

In your research you may have wanted to use automatic models for classification (like named entity recognition but also author identification, time period detection, sentiment analysis...), but you did not find any models that work well with your data topic or even language! So you think "surely, with the tools that are available now and free access to a GPU on Google Colab, I can train (or finetune) my own model!" Unfortunately you quickly hit the limits - not of the available tools, but of your own data. To use data-driven methods, you need qualitative data in a certain volume, which is not always available. Perhaps LLMs can lend a hand! In this notebook we will evaluate in how far LLMs can be used (if at all!) to create additional training data for named entity recognition and how that impacts the final model. So the task here is named entity recognition, but most of the methodology here applies to other tasks as well.

First, let's get some questions out of the way.

- Can't I just use an LLM to do the classification and cut out this whole middle process? What's the point?
> Yes you can! But there are cases where you may not want to, for instance: if you will have lots of data to run through your pipeline, it is cheaper (time/money/compute/environment-wise) to generate synthetic training data once and train a smaller classification model with that data compared to always use an LLM. Secondly, and more importantly, this notebook is intended to show what is methodologically possible and sound. How to train your own model, how to generate synthetic data based on in-domain data, and how to evaluate and compare models.
- So doesn't that mean we are effectively "distilling" the capabilities of the large LLM into a smaller model?
> Exactly, this is a form of distillation! By generating synthetic data with a large model, and optimizing (finetuning) a smaller model, we are teaching the smaller model to mimic the larger model. That way we end up with a small, efficient, and reusable model. This only works, of course, if the larger model can produce good quality data that our smaller model can learn from.
- Is an LLM the best way to label named entities in existing data? That seems very expensive when we might just use existing recognizers built into [spaCy](https://spacy.io/usage/linguistic-features#named-entities), [stanza](https://stanfordnlp.github.io/stanza/ner.html), [GLiNER](https://github.com/urchade/GLiNER), and so on.
> True! This notebook should be considered for its methodological inspiration. If you are interested exclusively in high-performance NER systems, then this notebook is **not** for you.

Legend

- ⚠️ => warning/strong recommendation. If not followed, it might lead to your code not working!
- ✅ => to-do recommendation/try it yourself
- 💡 => insight

---

Now, let's prepare our environment. It is tailored to run on Google Colab with a GPU enabled, which is a specialized piece of hardware that is much more efficient than a regular CPU when it comes to using neural networks.

> ⚠️ **To select a free GPU, go to "Runtime > Change runtime type" and select a GPU**!

In [1]:
!uv pip install --upgrade -q datasets==4.3.0 transformers==4.57.1 seqeval==1.2.2 triton==3.4 kernels accelerate==1.11.0 torch==2.8.0 numpy==2.2.6 ipywidgets
!uv pip uninstall -q torchvision torchaudio

In [2]:
import locale
import os

AVAIL_CORES = max(os.cpu_count() - 1, 1)
locale.getpreferredencoding = lambda: "UTF-8"

In [3]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    total_mem_gb = gpu.total_memory / (1024 ** 3)
    print(f"GPU: {gpu.name}")
    print(f"Total memory: {total_mem_gb:.2f} GB")
else:
    print("No GPU detected! Did you click 'Runtime > Change Runtime' and select a GPU?")

GPU: NVIDIA L40
Total memory: 44.39 GB


⚠️ This notebook relies heavily on the free Hugging Face infrastructure to upload/download our models and data. So before getting started you should create a Hugging Face (HF) account. Do not have an account yet? [Register now!](https://huggingface.co/join)

In the HF settings, go to Settings and find Access tokens (or go [here](https://huggingface.co/settings/tokens)). Create a new token. Under "User permissions" select all options under Repositories, or (less secure) simply select "Write". Scroll down and click "Create token". **Immediately copy this token!** If you close the modal, the Secret key will not be visible anymore for security reasons.

**Google Colab:** Once you have copied the code, go back to Colab (or this notebook), and on the left sidebar click on the key icon ("Secrets"). As a name, use `HF_TOKEN` (exactly!) and as a value, paste your copied Secret. In the future, and with this notebook if you are planning to run it, enable "Notebook access", indicating that this specific notebook can access that Secret key. Other people will never see your key.

**Locally:** Using the secret key/token that you retrieved above you can also log in on your own device. You can do that with `hf auth login` on the command line, following [this guide](https://huggingface.co/docs/huggingface_hub/en/guides/cli#hf-auth-login).

⚠️ **After entering the Secret in Colab or logging in on your own PC, you may need to restart the session (Runtime > Restart session).**

In [4]:
from huggingface_hub import whoami

# Set to False if you do not want to push models to the Hub
DO_USE_HUB = True  

whoami = whoami()
if whoami and "name" in whoami and whoami["type"] == "user":
    HF_ACCOUNT = whoami["name"]
    print(f"Logged in as {HF_ACCOUNT}!")
else:
    HF_ACCOUNT = None
    print("⚠️ No Hugging Face username found. If you want to run this notebook with all functionalities, you need a logged in account. Follow the steps above.")

Logged in as BramVanroy!


## Task description and dataset

As mentioned in the introduction, this notebook is mostly intended as methodological inspiration for your own work. We are not aiming to get the best-possible NER model but are running through a data augmentation pipeline that may (or may not) provide helpful additional data. Training on the augmented dataset could hopefully improve our model.

So as an example task, we want to train our own NER model and to do that we need data. A very well-known common dataset for this is the CoNLL 2003 dataset for English named entity recognition by Tjong Kim Sang and De Meulder. As with any task or dataset that you want to get your hands dirty with, it's a good idea to read the paper or dataset description. It often contains information, such as annotation guidelines or extra information, that can be helpful in making decisions about your approach.

✅ Read [the paper](https://aclanthology.org/W03-0419.pdf), [website](https://www.clips.uantwerpen.be/conll2003/ner) and [dataset description](https://huggingface.co/datasets/eriktks/conll2003)

Now let's peek at a data sample.

In [5]:
from datasets import load_dataset


conll_ds = load_dataset("BramVanroy/conll2003").select_columns(
    ["tokens", "ner_tags"]
).map(lambda toks: {"text": " ".join(toks)}, input_columns="tokens", num_proc=AVAIL_CORES)
conll_labels2idx = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}
conll_idx2labels = {v: k for k, v in conll_labels2idx.items()}

print(conll_ds)
for token, ner_tag in zip(conll_ds["train"][0]["tokens"], conll_ds["train"][0]["ner_tags"]):
    print(f"{token}\t{ner_tag}\t{conll_idx2labels[ner_tag]}")

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'text'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'text'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'text'],
        num_rows: 3453
    })
})
EU	3	B-ORG
rejects	0	O
German	7	B-MISC
call	0	O
to	0	O
boycott	0	O
British	7	B-MISC
lamb	0	O
.	0	O


💡 `German` and `British` are marked as B-MISC, which might be surprising. But, indeed, in the paper it is remarked on page 2:

> Named entity tagging of English and German
 training, development, and test data, was done by
 hand at the University of Antwerp. Mostly, MUC
 conventions were followed (Chinchor et al., 1999).
 **An extra named entity category called MISC was
 added to denote all names which are not already in
 the other categories. This includes adjectives, like
 *Italian*, and events, like *1000 Lakes Rally*, making it
 a very diverse category.**

 That is of course useful to know when manually inspecting the data and model output but also when generating synthetic data! If you want to dig deeper into the annotation guidelines, the CoNLL 2003 paper refers to "1999 Named Entity Recognition
 Task Definition" by Chinchor, et al. (1999). Unfortunately I have been unable to find that report but it seems to be a successor to [MUC-7](https://aclanthology.org/M98-1028/).

We can see that the dataset comes with a `train`, `validation` and `test` dataset split. `train` is used to train a model, and validation (also called "development" or "dev") is an auxiliary portion that can be used to optimize aspects of the training process (like hyperparameter tuning or intermediate evaluation), and the test set gives an indication of the model's performance on a held-out set that was never used at all in the training or optimization process.

Before we get our hands dirty with models and training, we want to make sure that this dataset is actually well-suited for our task. We already encountered one potential "quirk": the `MISC` category. But we could be able to just ignore that if we do not need it. But other checks are needed to. For starts, it is crucial to check these data splits for unwanted overlap, **also called "test data leakage"**. If a model is trained on a given sentence, it will naturally perform relatively well on that sentence compared to unseen data. During evaluation, that would lead to unfairly, higher scores! The whole point of testing on a held-out set is to see how well our trained model generalizes to unseen data. So, we have to check whether any data samples duplicated across the splits (and we'll also check within the splits).

In [6]:
from datasets import Dataset, DatasetDict

init_train_size = len(conll_ds["train"])
init_val_size = len(conll_ds["validation"])
init_test_size = len(conll_ds["test"])

init_train_texts = set(conll_ds["train"]["text"])
init_dev_texts = set(conll_ds["validation"]["text"])
init_test_texts = set(conll_ds["test"]["text"])

def deduplicate_dataset(dataset_to_dedup: Dataset | DatasetDict, verbose: bool = True) -> Dataset | DatasetDict:    
    texts = set()
    print_max = 50
    did_dedupe = False
    def deduplicate(text: str) -> bool:
        nonlocal print_max, did_dedupe
        if text in texts:
            if verbose:
                if print_max > 0:
                    print(f"Duplicate found: {text.strip()}")
                elif print_max == 0:
                    print("Even more duplicates found but not printing anymore warnings...")
            did_dedupe = True
            print_max -= 1
            return False
        else:
            texts.add(text)
            return True

    # Do NOT use muklti-processing here, as that would lead to each process having its own
    # 'texts' set, defeating the purpose of deduplication across the entire dataset
    deduped = dataset_to_dedup.filter(
        deduplicate,
        input_columns="text",
        num_proc=None,
        keep_in_memory=True,
    ).remove_columns("text")

    if verbose and not did_dedupe:
        print("No duplicates found!")

    return deduped

dedup_conll_ds = deduplicate_dataset(conll_ds)

train_size = len(dedup_conll_ds["train"])
val_size = len(dedup_conll_ds["validation"])
test_size = len(dedup_conll_ds["test"])

Filter:   0%|          | 0/14041 [00:00<?, ? examples/s]

Duplicate found: BEIJING 1996-08-22
Duplicate found: LONDON 1996-08-22
Duplicate found: LONDON 1996-08-22
Duplicate found: JERUSALEM 1996-08-22
Duplicate found: JERUSALEM 1996-08-22
Duplicate found: BAGHDAD 1996-08-22
Duplicate found: BAGHDAD 1996-08-22
Duplicate found: Reuters has not verified these stories and does not vouch for their accuracy .
Duplicate found: ATHENS 1996-08-22
Duplicate found: -- Dimitris Kontogiannis , Athens Newsroom +301 3311812-4
Duplicate found: It brought in 4,275 tonnes of British mutton , some 10 percent of overall imports .
Duplicate found: YORK , England 1996-08-22
Duplicate found: Results from the
Duplicate found: Second round
Duplicate found: LONDON 1996-08-22
Duplicate found: LONDON 1996-08-22
Duplicate found: LONDON 1996-08-22
Duplicate found: LONDON 1996-08-22
Duplicate found: LONDON 1996-08-22
Duplicate found: Second round
Duplicate found: 6-2
Duplicate found: Scorers :
Duplicate found: Gloria Bistrita - Ilie Lazar ( 32nd ) , Eugen Voica ( 84th )
D

Filter:   0%|          | 0/3250 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [7]:
print(f"Removed {init_train_size-train_size:,} samples from training set! ({init_train_size:,}->{train_size:,})")
print(f"Removed {init_val_size-val_size:,} samples from validation set! ({init_val_size:,}->{val_size:,})")
print(f"Removed {init_test_size-test_size:,} samples from test set! ({init_test_size:,}->{test_size:,})")
print()
print("Train & validation overlap:", len(init_train_texts.intersection(init_dev_texts)))
print("Train & test overlap:", len(init_train_texts.intersection(init_test_texts)))
print("Validation & test overlap:", len(init_dev_texts.intersection(init_test_texts)))

Removed 1,350 samples from training set! (14,041->12,691)
Removed 309 samples from validation set! (3,250->2,941)
Removed 354 samples from test set! (3,453->3,099)

Train & validation overlap: 129
Train & test overlap: 78
Validation & test overlap: 25


Wow! There are lots of duplicates in the data! And, more problematic, some of the data also seems quite noisy, with texts like `------------------------` and `W L PCT GB`. Indeed, one may want to also have a deeper look at the quality of the benchmark - which is exactly what recent work has done, e.g. [CleanCoNLL](https://aclanthology.org/2023.emnlp-main.533/) (2023; data requires custom processing) and [CoNLL-Sharp](https://aclanthology.org/2024.lrec-main.330/) (2024; data not available yet at time of writing). Considering this noise, and the broad but less useful category of `MISC`, we will look for a different dataset to use. After all - all we want is to train our NER system that can predict persons, organizations, and locations, so we simply need a relevant dataset to do so!


Creating manually annotated datasets is a time and labor-intensive endeavor so other recent projects have also called upon crowd annotation to create NER datasets. One such project is ["universal NER"](https://aclanthology.org/2024.naacl-long.243/) (UNER), a cross-lingual endeavor that adds NER tags to the Universal Dependencies treebanks. They only support `LOC`, `ORG`, `PER`, which we may consider as a good thing, seeing that `MISC` of CoNLL was relatively ill-defined. In their paper you will find more information regarding [annotation protocol](https://www.universalner.org/guidelines/), annotation agreement, and supported languages.

✅ Read [the paper](https://aclanthology.org/2024.naacl-long.243.pdf), [website](https://huggingface.co/universalner) and [dataset description](https://huggingface.co/datasets/universalner/universal_ner)

So from here on out, we will use UNER! Specifically the `en_ewt` subset, which is based on the same [UD subset](https://universaldependencies.org/treebanks/en_ewt/index.html) which in turn is based on the English Web Treebank. It consists of five genres of web media: weblogs, newsgroups, emails, reviews, and Yahoo! answers. As we will see, it is by no means perfect, though. Let's have a look.

In [8]:
orig_ds = load_dataset("BramVanroy/universal_ner", "en_ewt").select_columns(["text", "tokens", "ner_tags"])
ner_feats = orig_ds["train"].features["ner_tags"].feature

print("NER tags:", ner_feats)
print(orig_ds)
for token, ner_tag in zip(orig_ds["train"][0]["tokens"], orig_ds["train"][0]["ner_tags"]):
    print(f"{token}\t{ner_tag}\t{ner_feats.int2str(ner_tag)}")

NER tags: ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])
DatasetDict({
    train: Dataset({
        features: ['text', 'tokens', 'ner_tags'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['text', 'tokens', 'ner_tags'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['text', 'tokens', 'ner_tags'],
        num_rows: 2077
    })
})
Where	0	O
in	0	O
the	0	O
world	0	O
is	0	O
Iguazu	5	B-LOC
?	0	O


In the cell above, note `ner_feats`. In Hugging Face datasets we can define a `ClassLabel`, which is intended exactly for model training. It offers functionality such as easily converting labels to indices and the other way around, without having to create our own mappings. A small but useful utility! We will use it frequently later in this script so from here on out all our datasets must make use of the same `ner_feats` feature, which also means that all datasets must have the same underlying mapping of label-to-index and index-to-label.

In [9]:
init_train_size = len(orig_ds["train"])
init_val_size = len(orig_ds["validation"])
init_test_size = len(orig_ds["test"])

init_train_texts = set(orig_ds["train"]["text"])
init_dev_texts = set(orig_ds["validation"]["text"])
init_test_texts = set(orig_ds["test"]["text"])

dedup_ds = deduplicate_dataset(orig_ds)

train_size = len(dedup_ds["train"])
val_size = len(dedup_ds["validation"])
test_size = len(dedup_ds["test"])

print(dedup_ds)

Filter:   0%|          | 0/12543 [00:00<?, ? examples/s]

Duplicate found: Monday thru Friday
Duplicate found: I would recommend to start with something more simple like tin, pewter, aluminum or zinc, they all have a much more reasonable melting point.
Duplicate found: What do you think of these photos?
Duplicate found: Good luck!
Duplicate found: Good luck!
Duplicate found: No.
Duplicate found: Jim B
Duplicate found: Toronto.
Duplicate found: Good luck!
Duplicate found: Get a guinea pig.
Duplicate found: Do animals see images on a TV screen like humans do?
Duplicate found: thanks
Duplicate found: Thanks.
Duplicate found: No.
Duplicate found: Good luck!
Duplicate found: The horse is not in pain.
Duplicate found: He 's not starving.
Duplicate found: I do n't know.
Duplicate found: thanks
Duplicate found: THEN put the eggs in water treated a couple of days before with a water treatment and left open to "breathe".
Duplicate found: good luck!
Duplicate found: Michael:
Duplicate found: John -
Duplicate found: Michael Gapinski Account Vice Presiden

Filter:   0%|          | 0/2001 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 11734
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1878
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1912
    })
})


In [10]:
print(f"Removed {init_train_size-train_size:,} samples from training set! ({init_train_size:,}->{train_size:,})")
print(f"Removed {init_val_size-val_size:,} samples from validation set! ({init_val_size:,}->{val_size:,})")
print(f"Removed {init_test_size-test_size:,} samples from test set! ({init_test_size:,}->{test_size:,})")
print()
print("Train & validation overlap:", len(init_train_texts.intersection(init_dev_texts)))
print("Train & test overlap:", len(init_train_texts.intersection(init_test_texts)))
print("Validation & test overlap:", len(init_dev_texts.intersection(init_test_texts)))

Removed 809 samples from training set! (12,543->11,734)
Removed 123 samples from validation set! (2,001->1,878)
Removed 165 samples from test set! (2,077->1,912)

Train & validation overlap: 35
Train & test overlap: 46
Validation & test overlap: 27


Unfortunately data duplication and noise is quite common in benchmark dataset, as we see repeated here. Luckily we checked for it and removed duplicates. UNER is based on the Universal Dependencies treebanks, and in our case specifically on `ewt` (English Web Treebank), which is known to be noisy. After deduplication, we still have a reasonable amount of data to train on, so we can continue our experiments with this dataset.

After deduplicating we see that our training set has around 12,000 samples, which is not at all insignificant. In this notebook we would like to mimic a low-resource scenario to be able to measure the impact of generating synthetic training data, so we will artificially downsample our training set to a small size of 200 samples, simulating a scenario where you yourself have taken the time to make 200 manual annotations. We **do not change the validation and test sets** to ensure we can still do a full evaluation.

In [11]:
from copy import deepcopy
from transformers import set_seed


set_seed(42)
# You can change this size to a different value for experimenting with more/less data
DOWNSAMPLE_SIZE = 200
downsample_ds = deepcopy(dedup_ds)
downsample_ds["train"] = downsample_ds["train"].shuffle().take(DOWNSAMPLE_SIZE)
print(downsample_ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 200
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1878
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1912
    })
})


## Loading the base model

After analysing our data (deduplicating, quality control & annotation consistency checks (out of scope here)), we are ready to train the base model on our base model. As mentioned, the notebook is intended to inspire you and provide methodological guidance; we are not aiming to maximize any benchmark scores compared to the state-of-the-art.

For our base model, we will finetune a modern BERT-like model (aptly called [ModernBert](https://aclanthology.org/2025.acl-long.127/)) on the task of token classification. So to get started, we load the configuration and indicate that we have 7 labels (O and I/B for PER, ORG, LOC) and we load the model and tokenizer.

Parts of the code presented here are inspired by [this script](https://github.com/huggingface/transformers/blob/main/examples/pytorch/token-classification/run_ner.py).

In [12]:
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
)

BASE_MODEL_NAME = "microsoft/deberta-v3-base"

def initialize_model_and_tokenizer(model_name: str):
    # Load the model to finetune and tokenizer
    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=ner_feats.num_classes,
        finetuning_task="ner",
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
        add_prefix_space=config.model_type in ("bloom", "gpt2", "roberta", "deberta"),
    )
    # The piece of the pipeline that will glue (collate) samples together in a batch
    data_collator = DataCollatorForTokenClassification(tokenizer)

    model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)
    model.config.label2id = {label: label_idx for label_idx, label in enumerate(ner_feats.names)}
    model.config.id2label = {idx: label for label, idx in model.config.label2id.items()}
    return model, tokenizer, data_collator

model, tokenizer, data_collator = initialize_model_and_tokenizer(BASE_MODEL_NAME)

/vol1/bram/sshoc-llm-workshop-2025/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Tokens, subword tokens, numbers?!?

💡 As you may know, modern language models **do not work with linguistic tokens** as they are present in our dataset as the input data. Instead the concept of a "tokenizer" in the large language model world, is a word splitter that splits up words into parts that are statistically common. To the chagrin of many linguists, these parts are not necessarily linguistic/morphological in nature, as the example below shows. These parts are also, perhaps confusingly,  called tokens or subword units/tokens. `▁` is a special character that some tokenizers use to indicate that a character/token starts with a white-space in the real text, i.e. the start of a word (`Ġ` is another common one).

A language model does not work with strings or text under the hood but with numbers. The tokenizer therefore plays a crucial role: it chops up the given text into subword tokens, and then finds the corresponding identifier (number) of each subword token, as shown below. This sequence of numbers can then be fed into the model. Similarly, and somewhat simplified, the model will eventually output a number between 0 and 6 for each token which we can then convert to one of our 7 NER labels. Everything that happens inside the model is based on numbers!

In [13]:
tokenizer_test_string = "The bewildered marmot contemplated the existential implications of a lukewarm teacup."
print(tokenizer_test_string)
print(tokenizer.convert_ids_to_tokens(tokenizer.encode(tokenizer_test_string, add_special_tokens=False)))
print(tokenizer(tokenizer_test_string, add_special_tokens=False).input_ids)

The bewildered marmot contemplated the existential implications of a lukewarm teacup.
['▁The', '▁bewildered', '▁mar', 'mot', '▁contemplated', '▁the', '▁existential', '▁implications', '▁of', '▁a', '▁lukewarm', '▁teacup', '.']
[279, 47563, 16362, 41420, 25898, 262, 26474, 6634, 265, 266, 38563, 65118, 260]


As a minor optimization, we want to pre-calculate the longest sequence we can ever have in our dataset, counted in subword tokens. That way we do not have to add padding for the maximum sequence length of the model, which would slow processing. Another optimization could be to sort the data by length, so that each batch is somewhat homogenous in sequence length, leading to fewer unnecessary padding tokens, further improving throughput but with the downside that this may lead to model bias towards certain sequence lengths.

✅ Improved data loading is left to the more advanced reader. You can find inspiration in the [`group_by_length`](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer#transformers.TrainingArguments.group_by_length) training argument but it would require you to make the padding and `tokenize_and_align_labels` function to be called dynamically rather than as a pre-processing step.

In [14]:
def get_longest_seq_length(dataset: Dataset, _tokenizer):
    def get_token_length(examples: dict[str, list]):
        tokenized_inputs = _tokenizer(
            examples["tokens"],
            is_split_into_words=True,
            add_special_tokens=True,
        )
        return {"num_tokens": [len(input_ids) for input_ids in tokenized_inputs["input_ids"]]}
    _ds = dataset.map(get_token_length, num_proc=AVAIL_CORES, batched=True)
    max_seq_length = max(_ds["num_tokens"])
    return max_seq_length

max_seq_length = get_longest_seq_length(downsample_ds["train"], tokenizer)
print(f"Max sequence length in training data: {max_seq_length}")

Max sequence length in training data: 65


In the example above we saw how the tokenizer can automatically chop up our text into pieces and assign identifiers to each piece. We should now do that for the full dataset. There is one point of attention, though: our text is already split into tokens ("tokenized" in the non-LLM sense). So each input is not a single text but a list of its linguistic tokens.

A second important point is the preparation of the labels. Since our linguistic tokens may be split up further (like "marmot" above), we should decide on a strategy of what that means for our `PER`, `LOC`, `ORG` labels. After all, the model will output a prediction for each subword token! Assume that the `LOC` entity `Netherlands` is split up into `▁N`, `ether`, `lands`, it is common to only mark `▁N` as `LOC` (`B-LOC` to be precise), and ignore the other tokens during training. During inference, when we actually start using our model, we can then use the inverse strategy: every reconstructed linguistic token gets the entity label that was predicted for its first subword token. So since `▁N` was predicted as `B-LOC`, the whole linguistic token `Netherlands` will be tagged as such. An alternative approach is to label all those internal tokens as `I-LOC` but in my experience that confuses the model more and makes it harder to train the model.

In [15]:
def tokenize_and_align_labels(examples: dict[str, list], _tokenizer, _max_seq_len):
    tokenized_inputs = _tokenizer(
        examples["tokens"],
        padding="max_length",
        max_length=_max_seq_len,
        # The texts in our dataset are lists of words instead of full texts (with a label for each word).
        is_split_into_words=True,
    )
    labels = []
    for seq_idx, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=seq_idx)
        previous_word_idx = None
        label_ids = []
        # The word_idx is the word token (from our input tokens) that a subword token
        # corresponds to. This way we can map the NER labels to the subword tokens
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to -100
            # In other words, we only think it is important that the model learns to correctly
            # label the first subword token. In post-processing we can then apply the
            # suggested label for that first subword token to the whole word
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_ds = downsample_ds.map(
    tokenize_and_align_labels,
    batched=True,
    num_proc=AVAIL_CORES,
    fn_kwargs={"_tokenizer": tokenizer, "_max_seq_len": max_seq_length},
    desc="Running tokenizer on the dataset",
)

## About evaluation

In named entity recognition, it is common to evaluate the final system with precision, recall and F1 measures, which I have [written about extensively before](https://enrichment.ivdnt.org/training-materials/evaluation-of-part-of-speech-tagging/) in the context of part-of-speech tagging (POS). In NER shared tasks, entities are evaluated at the entity level and not at the token level. A model prediction is only considered correct if the whole entity span matches the real labels. In other words, only `"New"=B-LOC + "York"=I-LOC` would be correct and only marking `New` or only `York` will not yield any "partial points". Importantly, `O` ("outside", not an entity) is discarded in this kind of evaluation - only the entities matter. 

💡 So taking `ORG` entities as an example, metrics like precision, recall, and F1 can be summarized as:

- precision: out of all the times that the model predicts an `ORG` entity span, how many were correct? (can be cheated: if you only predict `ORG` for the cases that you are extremely sure about, you will get precision=1.0, even if you only make one `ORG` prediction)
- recall: out of all real `ORG` entities in the dataset, how many did the model correctly predict? (can be cheated: if you predict *everything* as `ORG` you will get recall=1.0)
- F1: harmonic mean of precision and recall which punishes imbalance between the two so that "cheating" is less likely (unlike a common arithmetic mean)

💡 Note that accuracy cannot be calculated on the entity-level because it is span-based. There is no set of pre-defined spans, in other words if we have a sequence like `[ORG, O, O, O, LOC]`, we assume ORG and LOC are their own spans, but there are no span annotations for non-entities. After all, where do we draw boundaries for `O`, can it have sub-spans, too? These boundaries are undefined and hence "true negatives" (correctly predicted spans that are not entities) are undefined, too. So for entity-level NER evaluation we do not use accuracy. However, looking at the predictions for individual subword tokens, we can calculate accuracy (tokens where predicted label = true label / number of tokens). But this is not useful: first and foremost the data is heavily biased towards non-entities `O`, so a model that only predicts `O` would have a high accuracy, and second it does not consider span boundaries so will not fully evaluate spans as hoped, e.g. misclassifying `I-LOC` will only count as one mistake but in reality it invalidates a full span.


In [16]:
from seqeval.metrics import f1_score, precision_score, recall_score

# Cheating with recall, predict all tags as PER
y_true = [['O', 'O', 'B-PER', 'O', 'O', 'B-PER', 'O']]
y_pred = [['B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER']]

print("Cheating recall")
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

# Cheating with precision, predict only one tag that we are very sure of as PER
y_true = [['B-PER', 'O', 'B-PER', 'O', 'B-PER', 'O', 'B-PER']]
y_pred = [['B-PER', 'O', 'O', 'O', 'O', 'O', 'O']]

print()
print("Cheating precision")
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

Cheating recall
Precision: 0.2857142857142857
Recall: 1.0
F1: 0.4444444444444445

Cheating precision
Precision: 1.0
Recall: 0.25
F1: 0.4


## Implementing the evaluation function

To automate our evaluation, we can make use of the tried-and-true [`seqeval`](https://github.com/chakki-works/seqeval) library, which implements entity-level precision, recall and F1. It defaults to micro-averaging the result, which means that class imbalance in the dataset (e.g. more `ORG` than `LOC`) will be reflected in the evaluation metric -- more true `ORG` entities in the dataset will mean that the performance on that class will be more important than the performance on the `LOC` class. In terms of calculation, it means that precision, recall, and F1 score are calculated across all samples. In macro-averaging every class is considered of equal value regardless of how many samples are actually of that class (so results for precision, recall, F1 and first calculated for each class and then averaged across all classes).

In [17]:
from typing import Any
import numpy as np
from seqeval.metrics import classification_report, accuracy_score


def get_seqeval_scores(labels: list[str], predictions: list[str], add_breakdown: bool = False) -> dict[str, Any]:
    """Get seqeval scores with optional breakdown per tag and macro average."""
    report = classification_report(
        y_true=labels,
        y_pred=predictions,
        output_dict=True,
    )
    report.pop("weighted avg")
    
    macro_avg = report.pop("macro avg")
    micro_avg = report.pop("micro avg")

    tag_scores = {
        type_name: {
            "precision": score["precision"],
            "recall": score["recall"],
            "f1": score["f1-score"],
            "support": score["support"],
        }
        for type_name, score in report.items()
    }

    result = {
        # Add f1 micro as the default, as a primary key metric,
        # so it can easily be used for model selection
        "f1": micro_avg["f1-score"],
        "precision": micro_avg["precision"],
        "recall": micro_avg["recall"],
        "accuracy": accuracy_score(y_true=labels, y_pred=predictions),
    }

    if add_breakdown:
        result["macro_average"] = macro_avg
        result["per_tag"] = tag_scores

    return result


def compute_seqeval_from_ints(labels: list[int], predictions: list[int], add_breakdown: bool = False) -> dict:
    if isinstance(labels, np.ndarray):
        labels = labels.tolist()
    if isinstance(predictions, np.ndarray):
        predictions = predictions.tolist()
        
    # Remove ignored index (special tokens are -100, cf. tokenize_and_align_labels function)
    true_predictions = [
        [ner_feats.int2str(p) for (p, l) in zip(seq_preds, seq_labels) if l != -100]
        for seq_preds, seq_labels in zip(predictions, labels)
    ]
    true_labels = [
        [ner_feats.int2str(l) for l in seq_labels if l != -100] for seq_labels in labels
    ]
    return get_seqeval_scores(true_labels, true_predictions, add_breakdown=add_breakdown)


def compute_metrics(model_output, add_breakdown: bool = False) -> dict:
    # Note: `ner_feats` is defined globally above so it should be the same for all datasets!
    predictions, labels = model_output
    # Find the predicted NER tag out of the logits (the index with the highest value)
    predictions = np.argmax(predictions, axis=2).tolist()
    labels = labels.tolist()    
    return compute_seqeval_from_ints(labels, predictions, add_breakdown=add_breakdown)

## Training the base model

With the dataset prepared and the necessary functions set up, we are now ready to train our base model. As said frequently now, we are not micro-optimizing every step of the way; we are using sensible default arguments here so performance gains can still be made with hyperparameter tuning.

✅ Hyperparameter tuning is left to you as an exercise! Typical hyperparameters to tune are batch size, learning rate, weight decay. Much of this process can be [automated](https://huggingface.co/docs/transformers/en/hpo_train), too!

In [18]:
from transformers import TrainingArguments, Trainer

TRAINED_MODELS = []

# Initialize our Trainer and train
down_output_dir = f"{BASE_MODEL_NAME.split('/')[-1]}-uner-down200"
training_args_defaults = {
    "seed": 42,
    "do_train": True,
    "do_eval": True,
    "num_train_epochs": 30,
    "eval_strategy": "steps",
    "eval_steps": 20,
    "save_strategy": "best",
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "learning_rate": 2.5e-5,
    "load_best_model_at_end": True,
    "push_to_hub": False,  # We will push manually later
    "report_to": "none",
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "save_total_limit": 1,
    "logging_steps": 1,
    "save_safetensors": True,
    "logging_first_step": True,
    # In a real-world scenario, remove this to speed up training
    # Added here for reproducibility of results in the workshop
    "full_determinism": True,
}
training_args = TrainingArguments(
    down_output_dir,
    **training_args_defaults,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [19]:
from pathlib import Path

train_result = trainer.train()

trainer.save_model()
down_hub_id = f"{HF_ACCOUNT}/{Path(down_output_dir).stem}"
if DO_USE_HUB and HF_ACCOUNT is not None:
    trainer.push_to_hub(down_hub_id)
    print(f"Model pushed to the Hub at https://hf.co/{down_hub_id}!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
20,0.335000,0.203005,0.033058,0.372093,0.017297,0.942314
40,0.162300,0.136441,0.303308,0.286538,0.322162,0.955325
60,0.084500,0.118987,0.389724,0.353251,0.434595,0.961167
80,0.023200,0.109418,0.475465,0.425043,0.539459,0.966444
100,0.015400,0.113958,0.551189,0.528246,0.576216,0.969062
120,0.014200,0.108143,0.621762,0.597015,0.648649,0.973010
140,0.013600,0.112569,0.631635,0.620438,0.643243,0.973534
160,0.006300,0.109917,0.668399,0.643644,0.695135,0.975105
180,0.005200,0.104917,0.678825,0.638704,0.724324,0.975790
200,0.003800,0.109081,0.667689,0.633366,0.705946,0.975145


/vol1/bram/sshoc-llm-workshop-2025/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model pushed to the Hub at https://hf.co/BramVanroy/deberta-v3-base-uner-down200!


In [20]:
from functools import partial

# Change compute_metrics to add breakdown so we have per-tag information
trainer.compute_metrics = partial(compute_metrics, add_breakdown=True)
eval_metrics = trainer.evaluate()

def print_metrics(metrics_dict: dict, eval_key: str = "eval_"):
    print(f"Accuracy: {metrics_dict[f"{eval_key}accuracy"]:.4f}")
    print(f"Precision: {metrics_dict[f"{eval_key}precision"]:.4f}")
    print(f"Recall: {metrics_dict[f"{eval_key}recall"]:.4f}")
    print(f"F1: {metrics_dict[f"{eval_key}f1"]:.4f}")
    print()
    for tag, tag_metrics in metrics_dict[f"{eval_key}per_tag"].items():
        print(f"Tag: {tag}")
        for metric_name, metric_value in tag_metrics.items():
            if metric_name != "support":
                print(f"  {metric_name}: {metric_value:.4f}")

print_metrics(eval_metrics)

TRAINED_MODELS.append((down_hub_id, down_output_dir, eval_metrics))

Accuracy: 0.9770
Precision: 0.6794
Recall: 0.7330
F1: 0.7051

Tag: LOC
  precision: 0.7348
  recall: 0.7569
  f1: 0.7457
Tag: ORG
  precision: 0.5314
  recall: 0.4911
  f1: 0.5104
Tag: PER
  precision: 0.7000
  recall: 0.8808
  f1: 0.7801


While the overall accuracy of the model appears high (0.94), this number is deceptive in the context of NER. As mentioned above, accuracy is calculated at the (subword) token level, which means that it is dominated by the overwhelmingly frequent `O` tags that are not part of any named entity. As a result, **a model can achieve a high accuracy score simply by predicting `O` correctly most of the time, even if it struggles with recognizing all other tags.** All in all, that is not very useful for us now.

The more informative metric in NER is the entity-level F1 score as calculated by `seqeval`, which in a preprocessing step extracts the entity spans (e.g. merging `B-PER I-PER` to a single `PER` entity). In our model so far, we find that the F1 score is relatively poor and the model is not great at our task: only 0.56. Despite providing a general overview of model performance, it is often useful to pick up the magnifying glass and inspect the model's performance on the individual entities, too. The model achieves a medicore F1 score of 0.65 but we get a better idea of why when we look at the scores for each tag.

Interestingly, `PER` entities are handled relatively well with an F1 of 0.78, suggesting that person names form a more stable or learnable category (perhaps expected for common person names). The other types are more error-prone, with `ORG` and `MISC` in particular proving difficult. The performance on the ORG tag illustrates this clearly: when the model predicts an ORG entity, it is correct only about half the time (p=0.46), meaning it frequently labels spans as ORG that should not have been.

In sum, our model is not doing great yet and despite decent performance on `PER` and `LOC`, other categories are left behind.

Since we may want to run multiple models in the rest of the notebook, let's create a wrapping function that does all of the above within one function call so that we do not have to keep repeating ourselves!

In [21]:
def train_ner(
    text_label_dataset: DatasetDict,
    model_name: str,
    output_dir: str,
    extra_training_args: dict = None,
) -> tuple[str, str]:
    extra_training_args = extra_training_args or {}
    model, tokenizer, data_collator = initialize_model_and_tokenizer(model_name)
    
    max_seq_length = get_longest_seq_length(text_label_dataset["train"], tokenizer)
    print(f"Max sequence length in training data: {max_seq_length}")
    tokenized_ds = text_label_dataset.map(
        tokenize_and_align_labels,
        fn_kwargs={"_tokenizer": tokenizer, "_max_seq_len": max_seq_length},
        batched=True,
        num_proc=AVAIL_CORES,
        desc="Running tokenizer on the dataset",
    )
    default_training_args = {
        "output_dir": output_dir,
        **training_args_defaults,
        **extra_training_args,
    }
    training_args = TrainingArguments(
        **default_training_args,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model()
    ft_model_name = f"{HF_ACCOUNT}/{Path(output_dir).stem}"
    if DO_USE_HUB and HF_ACCOUNT is not None:    
        trainer.push_to_hub(ft_model_name)
        print(f"Model pushed to the Hub at https://hf.co/{ft_model_name}!")

    trainer.compute_metrics = partial(compute_metrics, add_breakdown=True)
    eval_metrics = trainer.evaluate()
    print_metrics(eval_metrics)

    return ft_model_name, output_dir, eval_metrics


def print_model_outputs(model_outputs: list[tuple[str, str, dict]]):
    for model_output in model_outputs:
        hub_id, output_dir, eval_metrics = model_output
        print(f"Model: {hub_id} (stored at {output_dir})")
        print("Evaluation metrics:")
        for metric_name, metric_value in eval_metrics.items():
            if metric_name.startswith("eval_"):
                if isinstance(metric_value, float):
                    print(f"  {metric_name}: {metric_value:.4f}")
                else:
                    print(f"  {metric_name}: {metric_value}")
        print()

## Generating synthetic data (different notebook)

Because of the different software dependencies and the long runtime of generating synthetic data, a separate notebook is provided (`2-generating-synthetic-data.ipynb`). If you've had the opportunity to create your own dataset, you can load it below by replacing `None` in `synth_train_ds_name = None` by the name of your uploaded dataset, e.g. `"BramVanroy/synthetic-uner-ner-200-Qwen3-14B-AWQ"`. If you do not have your own data, you can use the `DEMO_MODE=true` flag which will use the prepared data that was already generated with that notebook.

## Preparing the mixed dataset

In [22]:
# Set to False if you ran the synthetic data generation notebook yourself (we will re-use prepared data)
# Set to True to use prepared data
DEMO_MODE = True

if DEMO_MODE:
    synth_train_ds_name = "BramVanroy/synthetic-uner-ner"
    synth_train_ds = load_dataset(synth_train_ds_name, "200", split="train")
    print("DEMO_MODE=True: in the following sections we will re-use data that I (Bram) have already prepared.")
else:
    synth_train_ds_name = None # ADD YOUR DATASET NAME HERE
    synth_train_ds = load_dataset(synth_train_ds_name, split="train")
    print(f"DEMO_MODE=False. Re-using prepared data from the synthetic data generation notebook that should have been uploaded to your account at https://hf.co/datasets/{synth_train_ds_name}.")

if synth_train_ds_name is None:
    raise ValueError("Please set 'synth_train_ds_name' to the name of your Hugging Face dataset containing the synthetic data, or enable DEMO_MODE.")
print(synth_train_ds)

DEMO_MODE=True: in the following sections we will re-use data that I (Bram) have already prepared.
Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 200
})


In [23]:
from datasets import concatenate_datasets, DatasetDict
from copy import deepcopy

# Concatenate the synthetic training data to the downsampled real training data
aug_ds = DatasetDict({
    "train": concatenate_datasets([deepcopy(synth_train_ds), deepcopy(downsample_ds["train"])]),
    "validation": deepcopy(downsample_ds["validation"]),
    "test": deepcopy(downsample_ds["test"]),
}).map(lambda toks: {"text": " ".join(toks)}, input_columns="tokens", num_proc=AVAIL_CORES)
dedup_aug_ds = deduplicate_dataset(aug_ds)
print(dedup_aug_ds)

Filter:   0%|          | 0/400 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1912 [00:00<?, ? examples/s]

No duplicates found!
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 400
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1878
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1912
    })
})


In [24]:
output_dir = f"{BASE_MODEL_NAME.split('/')[-1]}-uner-down-synth{len(dedup_aug_ds['train'])}"
aug_hub_id, aug_output_dir, aug_eval_metrics = train_ner(
    text_label_dataset=dedup_aug_ds,
    model_name=BASE_MODEL_NAME,
    output_dir=output_dir,
)
TRAINED_MODELS.append((aug_hub_id, aug_output_dir, aug_eval_metrics))

/vol1/bram/sshoc-llm-workshop-2025/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Max sequence length in training data: 65


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
20,0.405800,0.193371,0.055000,0.120000,0.035676,0.943321
40,0.163900,0.126635,0.330942,0.296741,0.374054,0.958226
60,0.057900,0.101623,0.484367,0.447706,0.527568,0.966927
80,0.062000,0.086415,0.668639,0.614687,0.732973,0.974702
100,0.052100,0.091924,0.701828,0.678788,0.726486,0.976595
120,0.015000,0.092161,0.731359,0.693127,0.774054,0.976716
140,0.073200,0.093146,0.744757,0.706796,0.787027,0.977925
160,0.012300,0.096468,0.724378,0.670968,0.787027,0.976434
180,0.019000,0.097485,0.728463,0.682075,0.781622,0.977079
200,0.031400,0.098180,0.739573,0.698367,0.785946,0.977441


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model pushed to the Hub at https://hf.co/BramVanroy/deberta-v3-base-uner-down-synth400!


Accuracy: 0.9782
Precision: 0.7075
Recall: 0.7924
F1: 0.7476

Tag: LOC
  precision: 0.7126
  recall: 0.7644
  f1: 0.7376
Tag: ORG
  precision: 0.5865
  recall: 0.6964
  f1: 0.6367
Tag: PER
  precision: 0.7953
  recall: 0.9007
  f1: 0.8447


In [25]:
print_model_outputs(TRAINED_MODELS)

Model: BramVanroy/deberta-v3-base-uner-down200 (stored at deberta-v3-base-uner-down200)
Evaluation metrics:
  eval_loss: 0.1172
  eval_f1: 0.7051
  eval_precision: 0.6794
  eval_recall: 0.7330
  eval_accuracy: 0.9770
  eval_macro_average: {'precision': 0.6553980511771689, 'recall': 0.7095861203782081, 'f1-score': 0.6787261662129644, 'support': 925}
  eval_per_tag: {'LOC': {'precision': 0.7347931873479319, 'recall': 0.7568922305764411, 'f1': 0.7456790123456791, 'support': 399}, 'ORG': {'precision': 0.5314009661835749, 'recall': 0.49107142857142855, 'f1': 0.5104408352668214, 'support': 224}, 'PER': {'precision': 0.7, 'recall': 0.8807947019867549, 'f1': 0.7800586510263929, 'support': 302}}
  eval_runtime: 1.9740
  eval_samples_per_second: 951.3860
  eval_steps_per_second: 29.8890

Model: BramVanroy/deberta-v3-base-uner-down-synth400 (stored at deberta-v3-base-uner-down-synth400)
Evaluation metrics:
  eval_loss: 0.1316
  eval_f1: 0.7476
  eval_precision: 0.7075
  eval_recall: 0.7924
  eval

## Interpretation

The results of the models we've trained so far should be somewhat similar to these:

```md
Model: BramVanroy/deberta-v3-base-uner-down200 (stored at deberta-v3-base-uner-down200)
Evaluation metrics:
  eval_loss: 0.1172
  eval_f1: 0.7051
  eval_precision: 0.6794
  eval_recall: 0.7330
  eval_accuracy: 0.9770
  eval_macro_average: {'precision': 0.6553980511771689, 'recall': 0.7095861203782081, 'f1-score': 0.6787261662129644, 'support': 925}
  eval_per_tag: {'LOC': {'precision': 0.7347931873479319, 'recall': 0.7568922305764411, 'f1': 0.7456790123456791, 'support': 399}, 'ORG': {'precision': 0.5314009661835749, 'recall': 0.49107142857142855, 'f1': 0.5104408352668214, 'support': 224}, 'PER': {'precision': 0.7, 'recall': 0.8807947019867549, 'f1': 0.7800586510263929, 'support': 302}}

Model: BramVanroy/deberta-v3-base-uner-down-synth400 (stored at deberta-v3-base-uner-down-synth400)
Evaluation metrics:
  eval_loss: 0.1316
  eval_f1: 0.7476
  eval_precision: 0.7075
  eval_recall: 0.7924
  eval_accuracy: 0.9782
  eval_macro_average: {'precision': 0.6981348750901136, 'recall': 0.787167283551041, 'f1-score': 0.7396869982944021, 'support': 925}
  eval_per_tag: {'LOC': {'precision': 0.7126168224299065, 'recall': 0.7644110275689223, 'f1': 0.7376058041112455, 'support': 399}, 'ORG': {'precision': 0.5864661654135338, 'recall': 0.6964285714285714, 'f1': 0.636734693877551, 'support': 224}, 'PER': {'precision': 0.7953216374269005, 'recall': 0.9006622516556292, 'f1': 0.8447204968944099, 'support': 302}}
```

By slightly augmenting the data with only 200 samples, we've bumped performance on all main metrics:

- precision: 0.6794 -> 0.7075
- recall: 0.7330 -> 0.7924
- f1: 0.7051 -> 0.7476

💡 Many researchers and developers stop here. **"Our model significantly improves over our baseline with an F1 increase of more than 4%."** (Note that statistically unjustified mention of 'significantly'!) But a keen eye would have already noticed that more interesting results are hidden beneath the surface! Indeed, while the overall picture seems to be in favor of the second model (higher p/r/f), it would seem that the performance per-tag warrants a more nuanced interpretation!

| Model                                | LOC F1 | ORG F1 | PER F1 |
| ------------------------------------ | :----: | :----: | :----: |
| `deberta-v3-base-uner-down200`       |  **0.746** |    0.510   |    0.780   |
| `deberta-v3-base-uner-down-synth400` |    0.738   |  **0.637** |  **0.845** |

From this table, a few important observations pop out:

- The performance of the supposedly improved model on the `LOC` category stays roughly the same, even slightly worse. So the improvements that we made are definitely not because of that tag
- `ORG` improves a lot! The second model's performance jumpts from 0.510 to 0.637. That is a very substantial (dare I say 'significant'?) relative improvement and tells us that the synthetic data has especially helped the model recognise organisations better (both in precision and recall).
- `PER` also improves clearly, going from 0.780 to 0.845, which again is a marked improvement, on top of an already decent score.

💡 Depending on your use-case, it can also be helpful to go deeper, and look into precision and recall specifically, rather than their aggregation in the F1 score. For instance, if you use the named entity tagger as a first step in your data processing, and later one want to, for instance, look up the tagged `PER` in another database, you may want to emphasize precision because misidentified entities will lead to error propagation in the rest of your work. Or the opposite, when scanning high volumes of data for `ORG` entities in the context of financial news (i.e. companies): missing an entity (and any news belonging to that entity) may lead to a big financial impact. In that case over-predicting (lower precision) in favor of higher recall may be what you want! **It always depends on your own usecase.**

Before drawing any final conclusions, let's first investigate whether any of these differences actually have any statistical weight to them. To do so, first we shall write a function that returns the predictions of a model.


In [26]:
def predict_ner(
    text_label_dataset: DatasetDict,
    model_name_or_dir: str,
    extra_training_args: dict = None,
    test_split: str = "validation",
) -> tuple[str, str]:
    extra_training_args = extra_training_args or {}
    model, tokenizer, data_collator = initialize_model_and_tokenizer(model_name_or_dir)
    
    max_seq_length = get_longest_seq_length(text_label_dataset["train"], tokenizer)
    print(f"Max sequence length in training data: {max_seq_length}")
    tokenized_ds = text_label_dataset.map(
        tokenize_and_align_labels,
        fn_kwargs={"_tokenizer": tokenizer, "_max_seq_len": max_seq_length},
        batched=True,
        num_proc=AVAIL_CORES,
        desc="Running tokenizer on the dataset",
    )
    default_training_args = {
        "output_dir": output_dir,
        **training_args_defaults,
        **extra_training_args,
        **{
            "do_train": False,
            "do_eval": False,
            "eval_strategy": "no",
            "save_strategy": "no",
        }
    }
    training_args = TrainingArguments(
        **default_training_args,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.compute_metrics = partial(compute_metrics, add_breakdown=True)
    eval_result = trainer.predict(tokenized_ds[test_split])
    print_metrics(eval_result.metrics, eval_key="test_")

    return eval_result

So now we can get the actual predictions for each sample. We are running these evaluations on the `validation` split because one might argue that we are still in the "model optimization phase": we are trying to figure out what the best model configuration is, before picking the final model that we wish to go with as our selected model. You can run the same code with `test` if you would like.

In [27]:
import numpy as np

try:
    output_dirs = [model_info[1] for model_info in TRAINED_MODELS]
except NameError:
    output_dirs = ["deberta-v3-base-uner-down-synth400", "deberta-v3-base-uner-down-down200"]
    
predict_results = []
labels = None
for output_dir in output_dirs:
    print(f"Predictions for model {output_dir}:")
    result = predict_ner(
        text_label_dataset=dedup_aug_ds,
        model_name_or_dir=output_dir,
        test_split="validation",
    )
    predict_results.append(result)

    if labels is not None and not np.array_equal(labels, result.label_ids):
        raise ValueError("⚠️ Warning: different label IDs found between models' predictions!")
    labels = result.label_ids

base_model_preds = predict_results[0].predictions
synth_aug_model_preds = predict_results[1].predictions

Predictions for model deberta-v3-base-uner-down200:
Max sequence length in training data: 65


Accuracy: 0.9770
Precision: 0.6794
Recall: 0.7330
F1: 0.7051

Tag: LOC
  precision: 0.7348
  recall: 0.7569
  f1: 0.7457
Tag: ORG
  precision: 0.5314
  recall: 0.4911
  f1: 0.5104
Tag: PER
  precision: 0.7000
  recall: 0.8808
  f1: 0.7801
Predictions for model deberta-v3-base-uner-down-synth400:
Max sequence length in training data: 65


Accuracy: 0.9782
Precision: 0.7075
Recall: 0.7924
F1: 0.7476

Tag: LOC
  precision: 0.7126
  recall: 0.7644
  f1: 0.7376
Tag: ORG
  precision: 0.5865
  recall: 0.6964
  f1: 0.6367
Tag: PER
  precision: 0.7953
  recall: 0.9007
  f1: 0.8447


Now that we have the individual predictions, we can go ahead and calculate how significant the difference between our models really is! To do so, we will use bootstrap resampling. The conceptual idea is that our test set (in our case `validation` set) is only a sample from a larger population of possible texts. In other words, it is just one out of many possible test sets!

To see how much our results depend on this specific sample, we create many (often thousand or more!) new "pseudo" test sets by sampling the existing one *with replacement*. This means that some sentences may appear multiple times, others not at all, and each resampled test set will be a slightly different slice of the data. For every one of these pseudo test sets we compute the evaluation metric again for both models. After repeating this a thousand times, we end up with a distribution of model scores and, importantly, **a distribution of the differences between the two models**.

![ci.png](https://cdn.mathblog.com/wp-content/uploads/2023/11/95-percent-confidence-interval.jpg)

(Picture by MathBlog)

This distribution of differences shows how much the observed difference between models could vary if we had tested on slightly different data drawn from the same source. If the value 0 (meaning "no difference between the models") falls outside the central 95% of that distribution, we can conclude that the difference is statistically significant and unlikely to be due to chance alone.

💡 But we can take it a step further and test a null hypothesis with a p-value. Under our null hypothesis, we assume that models A and B are equally good. To simulate this "null world," we recenter our bootstrap distribution of differences around zero by subtracting the observed mean difference. This recentered distribution represents what differences we would expect to see purely due to sampling variability if the models truly performed the same (model A = model B). In other words, in this new distribution model A = model B, and other differences that deviate from the mean (now at 0) are noise/"artefacts of chance".

We then ask: "in this null world where the models are equal, how often would we observe a difference as extreme as (or more extreme than) the actual difference we measured?" This proportion is our p-value. If extreme differences occur frequently in the recentered distribution (high p-value), it means our observed difference could easily have happened by chance, and we cannot conclude the models are truly different. Conversely, if such extreme differences are rare (low p-value, typically below 0.05), we have strong evidence that the performance difference is real and not just a result of sampling noise.

So basically, the null hypothesis cannot be rejected if the observed difference occurs frequently in the "null world". In that case, it's simply part of the natural noise surrounding the scenario where model A = model B. But if the observed difference occurs rarely under the null hypothesis, it's unlikely to be just random noise but rather a systematic, consistent phenomen, and we can conclude the models truly perform differently.

In [28]:
from functools import partial
import numpy as np
from numpy.typing import NDArray 
from tqdm.auto import tqdm


def get_value_from_metric(metrics_dict: dict[str, Any], metric: str) -> float:
    """Get value from metric string, e.g. "LOC_f1" or "f1"."""
    if "_" in metric:
        tag, metric_name = metric.split("_", 1)
        return metrics_dict["per_tag"][tag][metric_name]
    else:
        return metrics_dict[metric]


def bootstrap_ner_diff(
    preds_a: NDArray,
    preds_b: NDArray,
    gold_labels: NDArray,
    *,
    n_boot: int = 1000,
    alpha: float = 0.05,
    seed: int=42,
    # can be "f1", "precision", "recall", but also "LOC_f1", "PER_f1", etc. for per-tag metrics
    metric: str ="f1"
) -> dict[str, Any]:    
    # If you run this function often you may want to optimize it with multiprocessing
    # but Colab only has two cores in the free tier so might not be worth the overhead
    rng = np.random.default_rng(seed)
    get_score = partial(get_value_from_metric, metric=metric)
    
    # If needed, convert logits to predicted class indices
    if preds_a.ndim == 3:
        preds_a = np.argmax(preds_a, axis=2)
    if preds_b.ndim == 3:
        preds_b = np.argmax(preds_b, axis=2)

    # Actually observed difference
    score_a_obs = compute_seqeval_from_ints(predictions=preds_a, labels=gold_labels, add_breakdown=True)
    score_b_obs = compute_seqeval_from_ints(predictions=preds_b, labels=gold_labels, add_breakdown=True)
    delta_obs = get_score(score_a_obs) - get_score(score_b_obs)

    # number of sentences
    n = gold_labels.shape[0]

    # will store Δ_i = F1_A - F1_B for each bootstrap sample
    diffs = np.empty(n_boot, dtype=float)

    all_batch_idxs = rng.choice(n, size=(n_boot, n), replace=True)
    for bootstrap_idx, batch_idxs in tqdm(
        enumerate(all_batch_idxs),
        desc="Bootstrapping",
        total=n_boot,
        leave=False
    ):
        batch_labels = gold_labels[batch_idxs]
        batch_preds_a = preds_a[batch_idxs]
        batch_preds_b = preds_b[batch_idxs]

        # This is not efficient at all -- better to calculate TP/FP/FN once outside the loop
        # but I am running out of time! :-)
        scores_a = compute_seqeval_from_ints(
            predictions=batch_preds_a,
            labels=batch_labels,
            add_breakdown=True,
        )
        scores_b = compute_seqeval_from_ints(
            predictions=batch_preds_b,
            labels=batch_labels,
            add_breakdown=True,
        )
        diffs[bootstrap_idx] = get_score(scores_a) - get_score(scores_b)

    lower = np.percentile(diffs, 100 * (alpha / 2))
    upper = np.percentile(diffs, 100 * (1 - alpha / 2))

    # P-values are calculated with the null hypothesis that there is no difference between the two models
    # We center the bootstrap distribution around 0 by subtracting the observed difference, leading to
    # a distribution that simulates the null hypothesis (model A ~= model B)
    centered = diffs - delta_obs
    # In this hypothetical world, we want to see how often we get a difference
    # as extreme (or more extreme) as the one we observed originally. This gives us the p-value.
    # In other words, if this absolute value occurs frequently in the Null World, then that means
    # that it is not so surprising to see the observed difference when model A ~= model B,
    # so the p-value will be high, and we cannot reject the null hypothesis.
    # If this absolute value is very rare in the Null World, then the p-value will be low,
    # and we can reject the null hypothesis and conclude that model A is significantly different
    # from model B.
    # The test is two-tailed, so we look at both sides of the distribution, because we do not know
    # in advance whether model A will be better or worse than model B.
    p_value = (np.abs(centered) >= abs(delta_obs)).mean()

    return {
        "observed_score_a": float(get_score(score_a_obs)),  # metric(A) on full set
        "observed_score_b": float(get_score(score_b_obs)),  # metric(B) on full set
        "delta_obs": float(delta_obs),                      # metric(A) - metric(B) on full set
        "ci": (float(lower), float(upper)),                 # bootstrap CI for delta
        "p_value": float(p_value),                          # two-tailed p-value for H0: metric(A) == metric(B)
    }


In [29]:
bootstrap_ner_diff(
    base_model_preds,
    synth_aug_model_preds,
    labels,
    n_boot=1000,
    metric="f1"
)

Bootstrapping:   0%|          | 0/1000 [00:00<?, ?it/s]

{'observed_score_a': 0.7051482059282371,
 'observed_score_b': 0.7475777664456911,
 'delta_obs': -0.042429560517453946,
 'ci': (-0.06851622380544135, -0.019428678822631235),
 'p_value': 0.002}

So, what does this tell us? 

- `observed_score_a`: the initial F1 result we already had on the test set of model A (our base model)
- `observed_score_b`: the initial F1 result we already had on the test set of model B (our second model, with extra synthetic data)
- `delta_obs`: the observed difference between the two F1 scores above
- `ci`: the bootstrap 95% confidence interval on differences in performance. In other words, randomly resampling 1000 times will give us a difference between these two models within this range 95% of the time [-0.685, -0.0194]. So the base model is always worse than the synthetic model and, crucially, 0 does not lie in the confidence interval. That means that it is very unlikely that the models are equal (CI does not include a difference of 0) so the result is already statistically reliable
- `p_value`: as extra evidence we also calculate a two-tailed p-statistic. Here it means that under the null hypothesis (where the distribution of differences is recentered to 0), values as extreme or more extreme than `delta_obs` occur seldomly (about 0.2%) which means that it cannot be explained by "the null hypothesis world" - such `delta_obs` values do not seem to fit within that distribution. So it is very unlikely that `delta_obs` is just random sampling noise that would fit in the "null world" where model A = model B. So we reject the null hypothesis that the models are equal. Or put differently, **model A is significantly different from model B!**

In scientific reporting, you could phrase this result somewhat like this:

> We evaluated the effect of synthetic data on the task of NER in a low-resource scenario. For statistical scrutiny, we used paired bootstrap resampling (1,000 samples). The synthetic-data model significantly outperformed the base model (**0.748 vs. 0.705 F1; ΔF1 = 0.042**). The 95% bootstrap CI for the improvement was [0.019, 0.069] (0 not in the confidence interval), and the p-value under the paired bootstrap null was 0.002, confirming that the improvement is statistically significant.

For completeness' sake we should also do the same tests for each entity. That way we know in what respects the synthetic model actually improved over the baseline. This is crucial! We might be under the impression that our model is better across the board and start using the supposedly improved model. But is that fair? Let's look at the performance for each tag.

In [30]:
for ent_type in ("LOC", "ORG", "PER"):
    boot_result = bootstrap_ner_diff(
        base_model_preds,
        synth_aug_model_preds,
        labels,
        n_boot=1000,
        metric=f"{ent_type}_f1"
    )
    print(f"Bootstrap results for entity type {ent_type}:")
    print(boot_result)

Bootstrapping:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrap results for entity type LOC:
{'observed_score_a': 0.7456790123456791, 'observed_score_b': 0.7376058041112455, 'delta_obs': 0.008073208234433582, 'ci': (-0.03276219180953669, 0.043329868118998284), 'p_value': 0.673}


Bootstrapping:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrap results for entity type ORG:
{'observed_score_a': 0.5104408352668214, 'observed_score_b': 0.636734693877551, 'delta_obs': -0.12629385861072961, 'ci': (-0.1832113162069724, -0.07296220240931836), 'p_value': 0.0}


Bootstrapping:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrap results for entity type PER:
{'observed_score_a': 0.7800586510263929, 'observed_score_b': 0.8447204968944099, 'delta_obs': -0.06466184586801693, 'ci': (-0.09660945587133897, -0.03124184794366852), 'p_value': 0.0}


As the keen-eyed may already suspected, we can see significant gains in `PER` (~6.46%) and especially `ORG` (~12.63%). So if we are mostly interested in those tags, we can probably go ahead and use model B (trained with synthetic data in the mix). However, for `LOC` the observed difference is only ~0.8% and not significant at all (p=0.673). The confidence interval also contains 0 almost at the center of it, so it is likely that the observed difference is not statistically grounded but rather by chance. We cannot say that model B is significantly better than model A for the `LOC` type, so if you are mostly interested in tagging locations it does not really matter which of the two models you use. (Though in that case you would inspect in more detail the precision and recall of `LOC` for both models and decide which metric is most important to you.)

## Summary: what have we learned

In this and the other notebook we have talked about many aspects of training a NER system and augmenting datasets with synthetic data. Our focus lied on the methodology of training your own models, generating synthetic data with local models, and making statistically grounded comparisons between models. Interestingly, and perhaps coincidentally, we also found that in this particular case our synthetic data did help performance of the model a little bit. However, truth be told, the datasets are both still so small that I am sure that if you fiddle with the hyperparameters you can also come up with a scenario where the base model is actually better than the synthetic one! (Try it!) As mentioned before, getting the best system was not a priority in this notebook. I hope you have learned some new techniques when it comes to training encoder models, using LLMs to generate data, and **crucially also some reflection on what it means for one model to be better than another, and how to test it.** "Bigger number better?" Not always!

## Postscriptum: The upper bound

As a reference, and out of curiosity, let's also train a model on the full official training set. Obviously this is not possible in a typical low-resource scenario but it gives us an idea of an upper bound. Note that we train for 10 epochs, the same as specified in [the original Github repository](https://github.com/UniversalNER/uner_code). The deduplicated training set consists of almost 12000 samples, so roughly 30 times as many as we have trained with so far.

In [31]:
full_orig_ds = load_dataset("BramVanroy/universal_ner", "en_ewt").select_columns(["text", "tokens", "ner_tags"])
dedup_full_orig_ds = deduplicate_dataset(full_orig_ds, verbose=False)
print(dedup_full_orig_ds)

output_dir = f"{BASE_MODEL_NAME.split('/')[-1]}-uner-full"
hub_id, output_dir, eval_metrics = train_ner(
    text_label_dataset=dedup_full_orig_ds,
    model_name=BASE_MODEL_NAME,
    output_dir=output_dir,
    # Same number of epochs as described in the original repo
    # Evaluate every epoch
    extra_training_args={"num_train_epochs": 10, "eval_strategy": "epoch"},
)
TRAINED_MODELS.append((hub_id, output_dir, eval_metrics))

Filter:   0%|          | 0/12543 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2001 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 11734
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1878
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1912
    })
})


/vol1/bram/sshoc-llm-workshop-2025/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.001200,0.058293,0.801701,0.788703,0.815135,0.983725
2,0.000200,0.062464,0.813559,0.797508,0.830270,0.984612
3,0.000300,0.067386,0.811065,0.784057,0.840000,0.983766
4,0.000000,0.071473,0.828100,0.815514,0.841081,0.985377
5,0.003100,0.079353,0.829685,0.819620,0.840000,0.985619
6,0.000100,0.079562,0.832008,0.816008,0.848649,0.985417
7,0.000000,0.086819,0.826226,0.814932,0.837838,0.985538
8,0.000100,0.091121,0.829191,0.811594,0.847568,0.985659
9,0.000100,0.095746,0.832094,0.818182,0.846486,0.985699
10,0.000100,0.098085,0.831557,0.820189,0.843243,0.985619


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model pushed to the Hub at https://hf.co/BramVanroy/deberta-v3-base-uner-full!


Accuracy: 0.9856
Precision: 0.8202
Recall: 0.8432
F1: 0.8316

Tag: LOC
  precision: 0.8507
  recall: 0.8571
  f1: 0.8539
Tag: ORG
  precision: 0.6737
  recall: 0.7098
  f1: 0.6913
Tag: PER
  precision: 0.8914
  recall: 0.9238
  f1: 0.9073


In [32]:
print_model_outputs(TRAINED_MODELS)

Model: BramVanroy/deberta-v3-base-uner-down200 (stored at deberta-v3-base-uner-down200)
Evaluation metrics:
  eval_loss: 0.1172
  eval_f1: 0.7051
  eval_precision: 0.6794
  eval_recall: 0.7330
  eval_accuracy: 0.9770
  eval_macro_average: {'precision': 0.6553980511771689, 'recall': 0.7095861203782081, 'f1-score': 0.6787261662129644, 'support': 925}
  eval_per_tag: {'LOC': {'precision': 0.7347931873479319, 'recall': 0.7568922305764411, 'f1': 0.7456790123456791, 'support': 399}, 'ORG': {'precision': 0.5314009661835749, 'recall': 0.49107142857142855, 'f1': 0.5104408352668214, 'support': 224}, 'PER': {'precision': 0.7, 'recall': 0.8807947019867549, 'f1': 0.7800586510263929, 'support': 302}}
  eval_runtime: 1.9740
  eval_samples_per_second: 951.3860
  eval_steps_per_second: 29.8890

Model: BramVanroy/deberta-v3-base-uner-down-synth400 (stored at deberta-v3-base-uner-down-synth400)
Evaluation metrics:
  eval_loss: 0.1316
  eval_f1: 0.7476
  eval_precision: 0.7075
  eval_recall: 0.7924
  eval

Welp! It would seem that having 30x as much data *does* have a positive impact on the results! Training on so much data enables the model to outperform our previous attempts on all fronts, both globally (F1 of 0.8316 vs 0.7476) and for each tag individually.